# Laboratorio 4 — Validación y pipelines

Evaluar un modelo es **estimar el error que va a cometer sobre datos que no vio**, y ese
error depende tanto del modelo como de la preparación que se le aplicó a los datos. Este
laboratorio recorre las tres piezas:

1. **Hold-out** — apartar una parte de los datos antes de entrenar.
2. **Validación cruzada $K$-fold** — repartir los datos en $K$ partes y validar $K$ veces.
3. **`Pipeline` y `ColumnTransformer`** — encapsular imputación, codificación, escalado y
   modelo en un único objeto, de modo que la preparación se ajuste siempre sobre la parte
   de entrenamiento y nunca sobre el conjunto completo.

Hay **dieciséis ejercicios** intercalados. Cada uno se resuelve en pocas líneas.

Todo corre **sin archivos externos**: el conjunto de datos viene con `scikit-learn`.

## 0. Preparación

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.compose import ColumnTransformer
from sklearn.datasets import load_diabetes
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.model_selection import KFold, cross_val_score, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

plt.rcParams.update({"figure.figsize": (8, 3.5), "axes.grid": True, "grid.alpha": 0.3})
np.set_printoptions(precision=3, suppress=True)

### Los datos

`load_diabetes` es un conjunto de regresión que viene incluido en `scikit-learn`: 442
pacientes, 10 variables clínicas y una respuesta numérica que mide la progresión de la
enfermedad al cabo de un año. **No hay faltantes ni categóricas**, de modo que se puede
modelar directamente.

In [ ]:
datos = load_diabetes()
X, y = datos.data, datos.target

print(f"X: {X.shape}   ->  {X.shape[0]} observaciones, {X.shape[1]} atributos")
print(f"y: {y.shape}")
print(f"\nAtributos: {list(datos.feature_names)}")
print(f"\nRespuesta: media = {y.mean():.1f}, desvío = {y.std(ddof=1):.1f}, "
      f"rango = [{y.min():.0f}, {y.max():.0f}]")

Esta versión de los atributos **viene estandarizada de fábrica**: cada columna está centrada
y llevada a una escala común. Es una particularidad de este conjunto, no la regla, y es
justamente lo que la sección 3 deja de suponer.

Conviene retener el **desvío de la respuesta**: es el error que cometería un modelo que
predijera siempre la media. Cualquier modelo útil tiene que quedar por debajo.

In [ ]:
def rmse(y_real, y_predicho) -> float:
    """Raíz del error cuadrático medio, en las unidades de la respuesta."""
    return float(np.sqrt(mean_squared_error(y_real, y_predicho)))


print(f"Error de predecir siempre la media: {rmse(y, np.full_like(y, y.mean())):.2f}")

---

# 1. Hold-out

Se aparta un porcentaje de las observaciones **antes de entrenar**, se ajusta el modelo con
el resto y se mide el error sobre lo apartado.

### 1.1 Qué hace `train_test_split`

Antes de usarlo sobre datos reales conviene verlo sobre diez observaciones numeradas, para
entender **exactamente qué devuelve**.

In [ ]:
ejemplo = np.arange(10)          # observaciones 0 a 9
print("Todas:", ejemplo)

entrena, prueba = train_test_split(ejemplo, test_size=0.3, random_state=42)
print("Entrenamiento:", entrena, f"  ({len(entrena)} observaciones)")
print("Prueba       :", prueba, f"  ({len(prueba)} observaciones)")

Tres cosas para observar:

- La partición es **aleatoria**: no se toman las últimas tres, se sortean.
- Las dos partes son **disjuntas** y juntas cubren todo.
- `test_size=0.3` reservó 3 de las 10.

### Ejercicio 1

Correr la celda anterior con `random_state=0` y con `random_state=7`. **¿Cambian las
observaciones que caen en cada parte?** Comprobar además, con `np.intersect1d`, que
entrenamiento y prueba nunca comparten un índice.

In [ ]:
# TODO: repetir el split con random_state = 0 y con random_state = 7.
#       Imprimir las dos partes y, con np.intersect1d, los índices en común.
raise NotImplementedError()

### 1.2 El hold-out completo

Sobre los datos reales, el procedimiento tiene siempre los mismos cuatro pasos.

In [ ]:
# 1. Partir
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# 2. Entrenar SOLO con la parte de entrenamiento
modelo = LinearRegression()
modelo.fit(X_train, y_train)

# 3. Predecir sobre las dos partes
pred_train = modelo.predict(X_train)
pred_test = modelo.predict(X_test)

# 4. Medir
print(f"n entrenamiento = {len(X_train)}   n prueba = {len(X_test)}")
print(f"RMSE entrenamiento = {rmse(y_train, pred_train):.2f}")
print(f"RMSE prueba        = {rmse(y_test, pred_test):.2f}")
print(f"R2 prueba          = {r2_score(y_test, pred_test):.3f}")

El error sobre entrenamiento salió **apenas** más bajo que el de prueba. Con 442
observaciones y 10 atributos, un modelo lineal casi no tiene margen para adaptarse a las
particularidades de la muestra.

Eso cambia cuando hay pocas observaciones por atributo.

### Ejercicio 2

1. Repetir el hold-out (`test_size=0.3`, `random_state=42`) usando **solo las primeras 40
   observaciones**, y comparar el RMSE de entrenamiento con el de prueba.
2. Comparar la brecha en los dos casos. ¿En cuál el error de entrenamiento resulta
   claramente más optimista?
3. Con 442 observaciones, el error de prueba puede incluso salir **más bajo** que el de
   entrenamiento. ¿Contradice eso que el error de entrenamiento sea optimista?
4. **¿Cuál de los dos hay que reportar** como estimación del desempeño del modelo?
5. Con 40 observaciones y 10 atributos, ¿cuántas observaciones quedan por cada coeficiente
   estimado? Relacionarlo con lo observado.

In [ ]:
# TODO: repetir el hold-out (test_size=0.3, random_state=42) dos veces, con las
#       442 observaciones y con las primeras 40. Informar en cada caso el RMSE de
#       entrenamiento, el de prueba y cuántas observaciones hay por coeficiente.
raise NotImplementedError()

### Ejercicio 3

Repetir el hold-out con `test_size` igual a 0,1 · 0,2 · 0,3 · 0,5, manteniendo
`random_state=42`. Armar un `DataFrame` con el tamaño de cada parte y el RMSE de prueba.

**¿Qué le pasa al error estimado a medida que se reserva más?**

In [ ]:
# TODO: recorrer test_size en (0.1, 0.2, 0.3, 0.5) con random_state=42 y armar
#       un DataFrame con test_size, n_train, n_test y el RMSE de prueba.
raise NotImplementedError()

### 1.3 Qué quedó adentro del modelo ajustado

Ajustar deja los parámetros estimados guardados en el objeto. En `scikit-learn`, **todo
atributo que termina en guion bajo se creó durante el `fit`**: antes de ajustar no existe.
Para un modelo lineal son el vector de coeficientes y el intercepto.

In [ ]:
print("coef_     :", modelo.coef_.shape)
print(modelo.coef_.round(1))
print("intercept_:", round(float(modelo.intercept_), 2))

`coef_` trae un valor por columna de `X`, **en el mismo orden que las columnas**, y ese es
el único vínculo entre un número y el atributo al que corresponde: el arreglo no conserva
los nombres. Recuperarlos es responsabilidad de quien escribe el código.

### Ejercicio 4

1. Armar un `DataFrame` de dos columnas, `atributo` y `coeficiente`, uniendo `modelo.coef_`
   con `datos.feature_names`. Ordenarlo por valor absoluto del coeficiente, de mayor a
   menor.
2. Verificar que la predicción no es una caja negra: comprobar que
   `X_test @ modelo.coef_ + modelo.intercept_` coincide con `modelo.predict(X_test)`.
3. **¿Qué significa el intercepto acá?** Relacionar su valor con la media de la respuesta y
   con el hecho de que los atributos vienen centrados.
4. En este conjunto las columnas ya vienen en una escala común, y por eso los coeficientes
   son comparables entre sí. **¿Cuáles son los tres atributos de mayor peso?** ¿Qué habría
   que hacer antes de compararlos si cada columna viniera en sus unidades originales?
5. Reajustar el modelo con `random_state=0` en la partición y comparar los coeficientes con
   los anteriores. ¿Se mantienen estables el signo y el orden de magnitud?

In [ ]:
# TODO: (a) armar el DataFrame de coeficientes con datos.feature_names y ordenarlo
#           por valor absoluto;
#       (b) comprobar que X_test @ coef_ + intercept_ reproduce predict(X_test);
#       (c) reajustar con random_state=0 y comparar los coeficientes.
raise NotImplementedError()

---

# 2. Validación cruzada $K$-fold

En lugar de una sola partición, se reparten las observaciones en $K$ grupos —los **folds**—
y se entrena $K$ veces: cada vez, un fold distinto queda para validar y el resto para
entrenar. **Cada observación valida exactamente una vez.**

### 2.1 Qué hace `KFold`

`KFold` no parte los datos: **produce índices**. Otra vez sobre diez observaciones
numeradas, para verlo.

In [ ]:
kf = KFold(n_splits=5, shuffle=True, random_state=42)

for i, (idx_train, idx_val) in enumerate(kf.split(ejemplo), start=1):
    print(f"fold {i}:  entrena con {idx_train}   valida con {idx_val}")

`shuffle` viene en `False` por omisión, al revés que en `train_test_split`. Sin mezclar,
los folds respetan el orden del archivo.

### Ejercicio 5

Comprobar sobre la salida anterior las dos propiedades que definen a $K$-fold:

1. Cada observación aparece en el conjunto de **validación** de exactamente un fold.
   Concatenar los cinco arreglos de validación con `np.concatenate` y comparar su largo
   con la cantidad de valores distintos que devuelve `np.unique`.
2. En cada fold, entrenamiento y validación son disjuntos y juntos cubren todo.
   Verificarlo con `np.intersect1d` y `np.union1d`, fold por fold.

In [ ]:
# TODO: recorrer kf.split(ejemplo) y comprobar, sin mirar a ojo:
#       (a) que entrenamiento y validación no comparten índices (np.intersect1d);
#       (b) que juntos cubren las 10 observaciones (np.union1d);
#       (c) que las validaciones acumuladas son las 10, una sola vez cada una.
raise NotImplementedError()

### 2.2 El bucle a mano

Antes de usar la función que lo hace todo, conviene escribir el bucle: son cinco líneas y
son las que dejan claro **qué datos ve el modelo en cada vuelta**.

In [ ]:
kf = KFold(n_splits=5, shuffle=True, random_state=42)
puntajes = []

for idx_train, idx_val in kf.split(X):
    m = LinearRegression().fit(X[idx_train], y[idx_train])      # entrena con K-1 folds
    puntajes.append(rmse(y[idx_val], m.predict(X[idx_val])))    # mide sobre el que quedó

puntajes = np.array(puntajes)
print("RMSE de cada fold:", puntajes.round(2))
print(f"\nmedia  = {puntajes.mean():.2f}")
print(f"desvío = {puntajes.std(ddof=1):.2f}")

### 2.3 La versión corta: `cross_val_score`

`scikit-learn` hace ese bucle en una línea. Devuelve **un valor por fold**, no un promedio.

In [ ]:
kf = KFold(n_splits=5, shuffle=True, random_state=42)

# scoring="neg_root_mean_squared_error" devuelve el RMSE en negativo,
# porque scikit-learn asume que "más grande es mejor". Se le cambia el signo.
puntajes_sklearn = -cross_val_score(
    LinearRegression(), X, y, cv=kf, scoring="neg_root_mean_squared_error"
)

print("A mano         :", puntajes.round(4))
print("cross_val_score:", puntajes_sklearn.round(4))
print("\n¿Coinciden?", np.allclose(puntajes, puntajes_sklearn))

### Ejercicio 6

`cross_val_score` admite otras métricas con el argumento `scoring`.

1. Calcular la validación cruzada con `scoring="r2"` y reportar media y desvío.
2. ¿Por qué el $R^2$ **no** lleva el signo cambiado y el RMSE sí?

In [ ]:
# TODO: correr cross_val_score con scoring="r2", informar media y desvío,
#       y explicar en un comentario por qué acá no va el signo cambiado.
raise NotImplementedError()

### Ejercicio 7

Repetir la validación cruzada para $K$ = 2, 3, 5, 10 y 20, con
`KFold(n_splits=K, shuffle=True, random_state=42)`.

Armar el `DataFrame` `tabla_K` con: $K$, la media de los RMSE, el desvío entre folds, y
cuántos modelos hubo que entrenar.

In [ ]:
# TODO: recorrer K en (2, 3, 5, 10, 20) y para cada uno calcular la media de los
#       RMSE, el desvío entre folds y la cantidad de modelos entrenados.
raise NotImplementedError()

### Ejercicio 8

Leyendo `tabla_K`:

1. ¿Qué se mueve más al aumentar $K$: la media o el desvío entre folds? ¿Por qué?
2. Con $K=2$ cada modelo se entrena con la mitad de los datos. ¿Eso hace que el error
   estimado sea más alto o más bajo? ¿Por qué?
3. ¿Cuánto cuesta pasar de $K=5$ a $K=20$, en cantidad de modelos entrenados?

### Ejercicio 9

El argumento `shuffle` decide si los datos se mezclan antes de partirlos.

1. Comparar `KFold(n_splits=5, shuffle=False)` contra
   `KFold(n_splits=5, shuffle=True, random_state=42)`.
2. Con `shuffle=False`, ¿qué observaciones caen en el primer fold?
   (Mirarlo sobre `ejemplo`, con `n_splits=5`.)
3. ¿En qué situación **no mezclar** podría dar un resultado muy distinto?

In [ ]:
# TODO: (a) imprimir qué observaciones valida cada fold con shuffle=False sobre
#           `ejemplo`; (b) comparar la validación cruzada sobre X, y con y sin
#           mezcla, informando media y desvío en los dos casos.
raise NotImplementedError()

---

# 3. Preparar los datos sin fuga: `Pipeline`

Todo lo anterior supuso una matriz numérica y completa. Sobre datos crudos esa matriz no
existe: hay faltantes, hay columnas de texto y hay escalas dispares. Y la preparación que
resuelve eso **también tiene parámetros que se estiman a partir de datos** —una mediana,
una media, un desvío, la lista de niveles de una categórica—, de modo que dónde se estiman
decide si la validación mide lo que dice medir.

### 3.1 Un conjunto que no se puede modelar directamente

`load_diabetes(scaled=False)` devuelve los mismos datos **en sus unidades originales**, sin
estandarizar. Sobre esa versión se arma una tabla con dos dificultades habituales: una
columna categórica en texto y faltantes en cuatro columnas.

In [ ]:
crudo = load_diabetes(scaled=False, as_frame=True).frame
rng = np.random.default_rng(42)

tabla = crudo.drop(columns=["sex", "target"]).copy()
# El conjunto codifica el sexo como 1 y 2 sin documentar la correspondencia;
# pasarlo a texto lo convierte en la categórica que interesa tratar acá.
tabla.insert(1, "sexo", np.where(crudo["sex"] == 1, "grupo 1", "grupo 2"))
respuesta = crudo["target"].to_numpy()

# Faltantes introducidos deliberadamente, para tener qué imputar
for columna, cuantos in [("bmi", 35), ("bp", 20), ("s3", 12), ("sexo", 8)]:
    filas = rng.choice(len(tabla), size=cuantos, replace=False)
    tabla.loc[filas, columna] = np.nan

print(tabla.head(4))
print("\nFaltantes por columna:")
print(tabla.isna().sum()[lambda s: s > 0])
print("\nEscalas:")
print(tabla.describe().T[["min", "max"]])

Sobre esta tabla, el ajuste **no corre**:

In [ ]:
for columnas, descripcion in [(tabla.columns, "con la columna de texto"),
                              (tabla.columns.drop("sexo"), "solo las numéricas")]:
    try:
        LinearRegression().fit(tabla[columnas], respuesta)
    except ValueError as error:
        print(f"{descripcion}: {str(error).splitlines()[0]}")

### Ejercicio 10

1. Listar, con `tabla.dtypes`, qué columnas son numéricas y cuáles no. **Esa lista es la
   que después va a decidir qué rama del flujo recibe cada columna.**
2. Sobre la tabla de escalas: ¿cuántas veces más grande es el rango de `s1` que el de
   `s5`? ¿Qué problema trae eso para comparar coeficientes entre sí?
3. Descartar las filas con algún faltante (`tabla.dropna()`) y contar cuántas quedan.
   **¿Cuántas observaciones cuesta esa solución**, y por qué es peor de lo que sugiere
   mirar cada columna por separado?

In [ ]:
# TODO: (a) listar los tipos de cada columna y separar numéricas de categóricas;
#       (b) comparar el rango de s1 con el de s5;
#       (c) contar cuántas filas sobreviven a tabla.dropna().
raise NotImplementedError()

### 3.2 Los tres transformadores

Un **transformador** es un objeto con `fit` y `transform`: `fit` estima algo a partir de
los datos y lo guarda; `transform` aplica lo estimado. La distinción es la misma que separa
entrenar de predecir en un modelo.

**`StandardScaler`** lleva cada columna a media cero y desvío uno. Lo que aprende son
justamente esa media y ese desvío.

In [ ]:
juguete = pd.DataFrame({"horas": [100.0, 400.0, 700.0, 800.0],
                        "vibracion": [1.0, 2.0, 3.0, 2.0]})

escalador = StandardScaler().fit(juguete)
print("media de cada columna :", escalador.mean_)
print("desvío de cada columna:", escalador.scale_)
print("\nTransformado:\n", escalador.transform(juguete).round(3))

nuevo = pd.DataFrame({"horas": [250.0], "vibracion": [2.5]})
print("\nUna observación nueva:", escalador.transform(nuevo).round(3))

La observación nueva **no se estandariza con su propia media**: se le resta la media
aprendida del conjunto de ajuste. Es exactamente lo que va a pasar con cada fold de la
validación cruzada.

**`SimpleImputer`** reemplaza los faltantes por una constante estimada por columna
—`"mean"`, `"median"`, `"most_frequent"` o `"constant"`—, que queda guardada en
`statistics_`.

In [ ]:
con_huecos = pd.DataFrame({"temperatura": [60.0, np.nan, 71.0, 65.0, np.nan],
                           "presion": [5.0, 6.0, np.nan, 7.0, 6.0]})

imputador = SimpleImputer(strategy="median").fit(con_huecos)
print("mediana aprendida:", imputador.statistics_)
print("\nTransformado:\n", imputador.transform(con_huecos))

**`OneHotEncoder`** convierte una columna categórica en una columna binaria por nivel. Lo
que aprende es el conjunto de niveles.

In [ ]:
turnos = pd.DataFrame({"turno": ["mañana", "tarde", "noche",
                                 "mañana", "tarde"]})

codificador = OneHotEncoder(sparse_output=False).fit(turnos)
print("niveles :", codificador.categories_)
print("columnas:", codificador.get_feature_names_out())
print("\nTransformado:\n", codificador.transform(turnos))

In [ ]:
cod = OneHotEncoder(sparse_output=False, drop="first",
                    handle_unknown="ignore").fit(turnos)
no_visto = pd.DataFrame({"turno": ["rotativo"]})

print("Con drop='first' ->", cod.get_feature_names_out())
print("Nivel no visto   ->", cod.transform(no_visto))

Dos argumentos que no son decorativos:

- **`drop="first"`** omite un nivel, que pasa a ser la referencia. Con intercepto, incluir
  los tres niveles junto con la columna de unos produce una matriz singular: los tres suman
  exactamente la columna de unos.
- **`handle_unknown="ignore"`** define qué hacer cuando aparece un nivel que el ajuste no
  vio. Sin ese argumento, la transformación se interrumpe con un error, y **es una
  situación que la partición produce sola**: basta con que un nivel poco frecuente caiga
  entero del lado de validación.

### Ejercicio 11

El punto de todo lo anterior es **dónde se ajusta cada transformador**.

1. Partir `tabla` y `respuesta` con `test_size=0.2, random_state=42`.
2. Ajustar un `SimpleImputer(strategy="median")` sobre las columnas
   `["bmi", "bp", "s3"]` **de la parte de entrenamiento**, y otro sobre **la tabla
   completa**. Comparar `statistics_`.
3. Hacer lo mismo con `StandardScaler` sobre esas columnas ya imputadas y comparar `mean_`.
4. Las medianas de los dos ajustes difieren poco. **¿Por qué, aun así, la segunda forma
   está mal?** Responder en términos de qué información está disponible en el momento de
   predecir, no en términos del tamaño de la diferencia.
5. `fit_transform` es `fit` seguido de `transform`. **¿Sobre qué parte de los datos es
   legítimo llamarlo, y sobre cuál no?**

In [ ]:
# TODO: partir tabla/respuesta, ajustar SimpleImputer y StandardScaler sobre la
#       parte de entrenamiento y sobre la tabla completa, y comparar statistics_
#       y mean_ en los dos casos.
raise NotImplementedError()

### 3.3 `ColumnTransformer`: una rama por tipo de columna

Las columnas numéricas y las categóricas necesitan tratamientos distintos. Un
`ColumnTransformer` aplica un transformador diferente a cada grupo de columnas y pega los
resultados uno al lado del otro.

In [ ]:
numericas = ["age", "bmi", "bp", "s1", "s2", "s3", "s4", "s5", "s6"]
categoricas = ["sexo"]

rama_num = Pipeline([
    ("imputar", SimpleImputer(strategy="median")),
    ("escalar", StandardScaler()),
])

rama_cat = Pipeline([
    ("imputar", SimpleImputer(strategy="most_frequent")),
    ("codificar", OneHotEncoder(drop="first", handle_unknown="ignore")),
])

preparacion = ColumnTransformer([
    ("num", rama_num, numericas),
    ("cat", rama_cat, categoricas),
])

Cada rama es a su vez un `Pipeline`: **una lista de pares `(nombre, objeto)` que se aplican
en orden**. En la rama numérica, primero se imputa y después se escala: el escalador estima
media y desvío ignorando los faltantes, pero los deja pasar tal cual, y el modelo no los
admite. En la categórica, la moda cubre los huecos y recién entonces se codifica.

### 3.4 `Pipeline`: la preparación y el modelo, un solo objeto

El último paso de un `Pipeline` puede ser un modelo. El objeto resultante tiene `fit` y
`predict`, y se usa **exactamente igual que un `LinearRegression`**.

In [ ]:
flujo = Pipeline([
    ("preparar", preparacion),
    ("modelo", LinearRegression()),
])

T_train, T_test, r_train, r_test = train_test_split(
    tabla, respuesta, test_size=0.2, random_state=42
)

flujo.fit(T_train, r_train)
print(f"RMSE prueba = {rmse(r_test, flujo.predict(T_test)):.2f}")
print(f"R2 prueba   = {r2_score(r_test, flujo.predict(T_test)):.3f}")
print("\nColumnas que recibe el modelo:")
print(flujo.named_steps["preparar"].get_feature_names_out())

`flujo.fit(T_train, r_train)` ajusta **todo en cadena y con las observaciones de
entrenamiento únicamente**: las medianas, la moda, las medias y desvíos, los niveles de la
categórica y los coeficientes. `flujo.predict(T_test)` solo transforma con lo ya aprendido
y predice. La tabla de prueba entra cruda, con sus faltantes y su columna de texto.

**Con el flujo encapsulado, la fuga procedimental deja de ser expresable**: no hay manera
de escribir «ajustar el escalador sobre todo el conjunto y después partir» sin salirse
deliberadamente de la estructura.

### Ejercicio 12

Sobre el flujo anterior:

1. Cambiar `strategy="median"` por `"mean"` en la rama numérica y volver a medir. ¿Cuánto
   cambia el RMSE de prueba?
2. Quitar `handle_unknown="ignore"` del codificador y volver a ajustar. Con dos niveles y
   una partición al azar no falla; explicar **en qué situación sí fallaría**.
3. Invertir el orden de la rama numérica —escalar antes de imputar— y volver a medir.
   **No falla, y el RMSE no cambia.** Explicar por qué.
4. Repetir esa comparación de órdenes con
   `SimpleImputer(strategy="constant", fill_value=0)`. Ahí los dos órdenes **sí** dan
   resultados distintos. ¿Qué significa, en la columna original, rellenar con cero después
   de estandarizar?
5. `flujo.named_steps` da acceso a cada paso por su nombre. Recuperar el imputador de la
   rama numérica y mostrar sus `statistics_`. **¿Sobre qué observaciones se calcularon?**

In [ ]:
# TODO: comparar el RMSE de prueba del flujo cambiando:
#       (a) la estrategia de imputación (median / mean / constant con fill_value=0);
#       (b) el orden de los pasos de la rama numérica (imputar->escalar y al revés).
#       Después mostrar los statistics_ del imputador ajustado dentro del flujo.
raise NotImplementedError()

### 3.5 La validación envuelve al pipeline

El objeto que recibe `cross_val_score` es **el flujo completo**, no el modelo. En cada
fold, `scikit-learn` ajusta la cadena entera con las observaciones de ajuste de ese fold.

In [ ]:
kf = KFold(n_splits=5, shuffle=True, random_state=42)

errores_flujo = -cross_val_score(flujo, tabla, respuesta, cv=kf,
                                 scoring="neg_root_mean_squared_error")

print("RMSE de cada fold:", errores_flujo.round(2))
print(f"\n{errores_flujo.mean():.2f} ± {errores_flujo.std(ddof=1):.2f}")

**La validación cruzada envuelve al pipeline, nunca al revés.** Si la preparación se hace
antes, una sola vez y sobre la tabla completa, la validación cruzada mide un procedimiento
distinto del que se va a desplegar, y la cifra que reporta no corresponde a nada.

### Ejercicio 13

1. Construir una segunda versión del flujo **sin `StandardScaler`** en la rama numérica, y
   validarla con la misma partición. Comparar la media y el desvío de los RMSE con los del
   flujo escalado.
2. Los cinco valores coinciden salvo error de redondeo. **¿Por qué el escalado no cambia
   nada para mínimos cuadrados sin penalización?** Pensarlo en términos de qué le pasa al
   coeficiente cuando la columna se divide por una constante.
3. Enumerar tres razones por las que, aun así, conviene dejar el escalado en el flujo.

In [ ]:
# TODO: construir el mismo flujo sin StandardScaler, validarlo con la misma
#       partición y comparar los cinco RMSE con los del flujo escalado.
raise NotImplementedError()

### 3.6 Los coeficientes de un pipeline

Extraer los coeficientes de un flujo tiene un paso más que en la sección 1: hay que pedirle
al modelo su `coef_` y **al preparador los nombres de las columnas que produjo**, porque ya
no coinciden con las columnas de la tabla original.

In [ ]:
flujo.fit(tabla, respuesta)

nombres = flujo.named_steps["preparar"].get_feature_names_out()
betas = flujo.named_steps["modelo"].coef_

coeficientes_flujo = pd.DataFrame({"columna": nombres, "coeficiente": betas})
print(coeficientes_flujo.sort_values("coeficiente", key=abs, ascending=False)
      .round(2).to_string(index=False))
print("\nintercepto:", round(float(flujo.named_steps["modelo"].intercept_), 2))

El prefijo indica de qué rama del `ColumnTransformer` salió cada columna.
`cat__sexo_grupo 2` es la única columna que quedó de la categórica: el nivel `grupo 1` es
la referencia que `drop="first"` omitió.

### Ejercicio 14

1. Interpretar el coeficiente de `num__bmi`: **un desvío estándar más de índice de masa
   corporal, ¿cuánto mueve la progresión estimada?**
2. Interpretar `cat__sexo_grupo 2`. ¿Respecto de qué se mide esa diferencia?
3. Repetir la extracción sobre el flujo **sin escalar** del ejercicio 13 y poner las dos
   columnas de coeficientes lado a lado. Los dos modelos predicen exactamente lo mismo,
   pero los números no se parecen. **¿Cuál de las dos tablas admite comparar la importancia
   relativa de los atributos, y por qué?**
4. Verificar la relación entre ambos: el coeficiente sin escalar de una columna es el
   coeficiente escalado dividido por el desvío de esa columna.
5. **¿Por qué el intercepto de este flujo no coincide con la media de la respuesta**, si
   las columnas numéricas están centradas?

In [ ]:
# TODO: (a) extraer los coeficientes del flujo sin escalar y ponerlos al lado de
#           los del flujo escalado; (b) verificar que el coeficiente sin escalar es
#           el escalado dividido por scale_ del escalador; (c) explicar el valor del
#           intercepto.
raise NotImplementedError()

---

# 4. Las dos estrategias de validación, comparadas

### Ejercicio 15

Poner las dos en el mismo cuadro.

1. Calcular el hold-out con `test_size=0.2` para `random_state` de 0 a 19 (20 valores).
2. Calcular la validación cruzada con $K=5$ para `random_state` de 0 a 19 (20 valores,
   cada uno la media de sus 5 folds).
3. Graficar los dos conjuntos de valores en un `boxplot` lado a lado.

**¿Cuál de los dos procedimientos da un resultado más estable al cambiar la semilla?**

In [ ]:
# TODO: para random_state de 0 a 19 calcular el RMSE de un hold-out 80/20 y la
#       media de un K-fold con K=5. Informar media y desvío de cada conjunto de
#       20 valores y graficarlos en un boxplot lado a lado.
raise NotImplementedError()

### Ejercicio 16

Completar la tabla con lo observado en el laboratorio.

| Criterio | Hold-out | $K$-fold |
|---|---|---|
| Modelos que entrena | | |
| Estabilidad del resultado | | |
| Aprovechamiento de los datos | | |
| Costo de cómputo | | |
| Cuándo conviene | | |

Y responder en dos o tres líneas cada una:

1. Si el conjunto tuviera 50 observaciones en lugar de 442, **¿cuál de los dos
   procedimientos usarías?** Justificar con lo medido en el ejercicio 15.
2. Si entrenar el modelo llevara dos horas, ¿cambiaría la respuesta?
3. Un informe reporta «el modelo tiene un RMSE de 54,2». **¿Qué información falta**
   para que esa cifra sea interpretable?

---

## Cierre

Lo que este laboratorio deja instalado:

- **`train_test_split`** parte una vez, al azar. Es rápido y su resultado depende del
  sorteo.
- **`KFold` + `cross_val_score`** parten $K$ veces, usan todas las observaciones para
  validar y devuelven $K$ números.
- **Nunca se reporta la media sola.** Un error estimado sin su variabilidad no dice si la
  diferencia entre dos modelos es real o es el sorteo.
- **El error medido sobre los datos de entrenamiento no sirve** como estimación del
  desempeño: en promedio es optimista.
- **Un imputador, un escalador y un codificador estiman parámetros igual que un modelo.**
  Por eso se ajustan sobre la parte de entrenamiento y se aplican, ya ajustados, sobre el
  resto.
- **El `Pipeline` no es una comodidad de biblioteca**: es la estructura que impide escribir
  la fuga. La validación cruzada envuelve al flujo completo, nunca al revés.
- **Los coeficientes se leen con sus nombres.** `coef_` es un arreglo sin etiquetas, y en
  un flujo los nombres los da `get_feature_names_out` del preparador.